# Hitter WAR Pipeline - Complete Workflow

**Purpose:** Train hitter WAR model from scratch and generate 2025 projections

**Last Updated:** 2025-10-06

---

## Pipeline Overview
1. Load historical data (2016-2024) for training
2. Run sklearn pipeline (filters → transformers → features)
3. Train unified ensemble model (single model for all positions)
4. Generate 2025 predictions with ROS projections
5. Validate performance (MAE, R², residuals)
6. Feature importance analysis
7. Error analysis by position
8. Save model and predictions

**Key Design:** Unlike pitchers (3 separate models), hitters use 1 unified model. Positional differences are handled by the Positional_WAR feature.

In [18]:
# Cell 1: Imports and Setup

import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Add project root to path
project_root = Path('.').absolute().parent.parent.parent
sys.path.insert(0, str(project_root))

from new_pipeline.notebooks.shared.pipeline_runner import (
    load_historical_data,
    load_current_season_data,
    run_data_pipeline,
    generate_predictions,
    calculate_metrics,
    split_by_position
)
from new_pipeline.notebooks.shared.plotting_utils import (
    create_actual_vs_predicted,
    create_residual_plot,
    create_feature_importance
)
from new_pipeline.notebooks.shared.analysis_utils import (
    calculate_elite_performance,
    analyze_errors_by_group
)
from new_pipeline.models.current_season import HitterEnsemble
from new_pipeline.common.constants import HITTER_MODEL_FEATURES

print("Imports successful!")
print(f"Hitter features: {len(HITTER_MODEL_FEATURES)}")

Imports successful!
Hitter features: 9


In [19]:
# Cell 2: Load Historical Training Data

print("Loading historical hitter data (2016-2024)...")

hitter_historical = load_historical_data(
    player_type='hitter',
    years=range(2016, 2025)
)

print(f"\nLoaded {len(hitter_historical)} hitter-seasons")
print(f"Years: {sorted(hitter_historical['Year'].unique())}")
print(f"\nSample columns: {list(hitter_historical.columns[:10])}")

Loading historical hitter data (2016-2024)...

Loaded 5760 hitter-seasons
Years: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Sample columns: ['Name', 'Team', 'G', 'PA', 'HR', 'R', 'RBI', 'SB', 'BB%', 'K%']


In [20]:
# Cell 3: Run Data Pipeline

print("Running sklearn pipeline...")
print("Steps: Filters → Feature Loading → Imputation → Validation → Selection → Normalization")

hitter_processed = run_data_pipeline(
    hitter_historical,
    player_type='hitter'
)

print(f"\nPipeline complete!")
print(f"Processed {len(hitter_processed)} qualified hitters")
print(f"Features: {len(HITTER_MODEL_FEATURES)}")
print(f"\nFeature list: {HITTER_MODEL_FEATURES}")
print(f"\nTarget: WAR_per_600 (range: {hitter_processed['WAR_per_600'].min():.2f} to {hitter_processed['WAR_per_600'].max():.2f})")

15:00:11 - new_pipeline.common.transformers.filters - INFO - PAFilter: Removed 1524 hitters with < 75 PA (full season)


Running sklearn pipeline...
Steps: Filters → Feature Loading → Imputation → Validation → Selection → Normalization


15:00:11 - new_pipeline.common.transformers.age_enricher - INFO - AgeEnricher: Loaded Age for 2088 hitters
15:00:11 - new_pipeline.common.transformers.age_enricher - INFO - AgeEnricher: Added Age column (range: 20-45)
15:00:11 - new_pipeline.common.transformers.hitter_features - INFO - Loading hitter features...
15:00:16 - new_pipeline.common.transformers.hitter_features - INFO - Loaded 11 hitter feature sets (33 total columns)
15:00:17 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Learned replacement values for 28 features
15:00:17 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Imputed 70 missing values
15:00:17 - new_pipeline.common.transformers.validators - WARNING - FeatureValidator found issues:
  - Feature 'K%' range [3.09, 51.25] outside expected [0, 50]
  - Feature 'AVG' range [0.09, 0.38] outside expected [0.1, 0.4]
  - Feature 'OBP' range [0.10, 0.49] outside expected [0.2, 0.5]
  - Feature 'SLG' range [0.13, 0.73] ou


Pipeline complete!
Processed 4236 qualified hitters
Features: 9

Feature list: ['K%', 'BB%', 'AVG', 'OBP', 'SLG', 'GDP', 'Positional_WAR', 'Enhanced_Baserunning', 'Enhanced_Defense']

Target: WAR_per_600 (range: -5.00 to 11.93)


In [21]:
# Cell 4: Position Distribution Analysis

print("Analyzing position distribution...")

# Note: Position info needs to be added to pipeline or loaded separately
# For now, show distribution if Primary_Position column exists
if 'Primary_Position' in hitter_processed.columns:
    position_splits = split_by_position(hitter_processed)
    
    print(f"\nInfielders (IF): {len(position_splits['IF'])} ({len(position_splits['IF'])/len(hitter_processed)*100:.1f}%)")
    print(f"Outfielders (OF): {len(position_splits['OF'])} ({len(position_splits['OF'])/len(hitter_processed)*100:.1f}%)")
    print(f"Catchers (C): {len(position_splits['C'])} ({len(position_splits['C'])/len(hitter_processed)*100:.1f}%)")
    print(f"Designated Hitters (DH): {len(position_splits['DH'])} ({len(position_splits['DH'])/len(hitter_processed)*100:.1f}%)")
else:
    print("\nNote: Position information not available in processed data")
    print("All hitters will be trained with unified model (position handled by Positional_WAR feature)")

Analyzing position distribution...

Infielders (IF): 1684 (39.8%)
Outfielders (OF): 1774 (41.9%)
Catchers (C): 533 (12.6%)
Designated Hitters (DH): 0 (0.0%)


In [22]:
# Cell 5: Prepare Training Data

print("Preparing training data...")

# Extract features and target
X_train = hitter_processed[HITTER_MODEL_FEATURES].values
y_train = hitter_processed['WAR_per_600'].values

print(f"Training data shape: {X_train.shape}")
print(f"Target shape: {y_train.shape}")
print(f"\nUnified model will be trained on all {len(X_train)} hitters")
print("Position differences handled by Positional_WAR feature")

Preparing training data...
Training data shape: (4236, 9)
Target shape: (4236,)

Unified model will be trained on all 4236 hitters
Position differences handled by Positional_WAR feature


In [23]:
# Cell 6: Train Unified Ensemble Model

print("Training hitter unified ensemble...")
print("  Single model: RandomForest + Keras + MultiQuantileHistGB")
print("\nThis may take 2-3 minutes...\n")

hitter_model = HitterEnsemble()
hitter_model.fit(X_train, y_train)

print("\nTraining complete!")
print("Model trained: Unified ensemble (3 sub-models)")
print("Applies to all positions (IF, OF, C, DH)")

15:00:17 - new_pipeline.models.current_season.hitter_ensemble - INFO - Training hitter ensemble (4236 samples)...
15:00:17 - new_pipeline.models.current_season.hitter_ensemble - INFO -   Training ExtraTrees...


Training hitter unified ensemble...
  Single model: RandomForest + Keras + MultiQuantileHistGB

This may take 2-3 minutes...



15:00:17 - new_pipeline.models.current_season.hitter_ensemble - INFO -   Training Keras (AdamW + Swish + BatchNorm)...
c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.




Epoch 52: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.

Epoch 62: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.
Epoch 67: early stopping
Restoring model weights from the end of the best epoch: 42.


15:00:50 - new_pipeline.models.current_season.hitter_ensemble - INFO -   Training MultiQuantileHistGB...
15:00:58 - new_pipeline.models.current_season.hitter_ensemble - INFO -   Hitter ensemble training complete



Training complete!
Model trained: Unified ensemble (3 sub-models)
Applies to all positions (IF, OF, C, DH)


In [24]:
# Cell 7: Training Set Validation

print("Validating on training data...")

y_pred_train = hitter_model.predict(X_train)

metrics = calculate_metrics(y_train, y_pred_train)

print("\n" + "="*50)
print("TRAINING METRICS")
print("="*50)
print(f"MAE:  {metrics['MAE']:.3f}")
print(f"RMSE: {metrics['RMSE']:.3f}")
print(f"R²:   {metrics['R²']:.3f}")

# Elite hitter performance
elite_metrics = calculate_elite_performance(y_train, y_pred_train, threshold=5.0)
print(f"\nElite (>5 WAR) MAE: {elite_metrics['elite_MAE']:.3f} ({elite_metrics['elite_count']} hitters)")

print("="*50)

Validating on training data...

TRAINING METRICS
MAE:  0.822
RMSE: 1.063
R²:   0.810

Elite (>5 WAR) MAE: 0.818 (285 hitters)


In [25]:
# DIAGNOSTIC 2: Verify Multi-Year Feature Consistency Bug
print("DIAGNOSTIC 2: Testing if Enhanced features are constant across years")
print("="*80)
print()

# Find players who appear in multiple years
player_year_counts = hitter_processed.groupby('MLBAMID').size()
multi_year_players = player_year_counts[player_year_counts >= 3].index[:5]  # Get 5 players with 3+ years

print("Checking 5 players who appear in 3+ years:")
print("If Enhanced_Baserunning/Enhanced_Defense are CONSTANT across years, the bug is confirmed.")
print()

for mlbamid in multi_year_players:
    player_data = hitter_processed[hitter_processed['MLBAMID'] == mlbamid][
        ['Name', 'Year', 'WAR', 'WAR_per_600', 'Enhanced_Baserunning', 'Enhanced_Defense', 'K%', 'BB%']
    ].sort_values('Year')
    
    if len(player_data) >= 3:
        name = player_data.iloc[0]['Name']
        print(f"{name} (MLBAMID {mlbamid}):")
        print(player_data.to_string(index=False))
        
        # Check variance
        br_variance = player_data['Enhanced_Baserunning'].var()
        def_variance = player_data['Enhanced_Defense'].var()
        kpct_variance = player_data['K%'].var()
        
        print(f"  Enhanced_Baserunning variance: {br_variance:.6f} {'CONSTANT ACROSS YEARS!' if br_variance < 0.001 else 'varies by year (correct)'}")
        print(f"  Enhanced_Defense variance:     {def_variance:.6f} {'CONSTANT ACROSS YEARS!' if def_variance < 0.001 else 'varies by year (correct)'}")
        print(f"  K% variance (for comparison):  {kpct_variance:.6f} (should vary by year)")
        print()

print("="*80)
print("Expected: Enhanced features should VARY by year (just like K%).")
print("Actual: If variance is ~0.0, features are CONSTANT = BUG CONFIRMED.")
print("="*80)

DIAGNOSTIC 2: Testing if Enhanced features are constant across years

Checking 5 players who appear in 3+ years:
If Enhanced_Baserunning/Enhanced_Defense are CONSTANT across years, the bug is confirmed.

Adrian Beltré (MLBAMID 134181):
         Name  Year      WAR  WAR_per_600  Enhanced_Baserunning  Enhanced_Defense        K%       BB%
Adrian Beltré  2016 5.158574     4.836163             -0.463046         15.000000 10.312500  7.500000
Adrian Beltré  2017 2.559342     3.947572             -0.641253         13.793637 13.367609 10.025707
Adrian Beltré  2018 1.150791     1.435498             -0.587224         13.866537 19.958420  7.068607
  Enhanced_Baserunning variance: 0.008350 varies by year (correct)
  Enhanced_Defense variance:     0.457561 varies by year (correct)
  K% variance (for comparison):  24.302708 (should vary by year)

Victor Martinez (MLBAMID 400121):
           Name  Year       WAR  WAR_per_600  Enhanced_Baserunning  Enhanced_Defense        K%      BB%
Victor Martinez  2

In [26]:
# Cell 8: Load 2025 Data for Predictions

print("Loading 2025 current season data...")

hitter_2025_raw = load_current_season_data('hitter', year=2025)

print(f"Loaded {len(hitter_2025_raw)} hitters (raw)")

# Run pipeline
print("\nProcessing through pipeline...")
hitter_2025_processed = run_data_pipeline(hitter_2025_raw, player_type='hitter')

print(f"Processed {len(hitter_2025_processed)} qualified hitters")

15:00:59 - new_pipeline.common.transformers.filters - INFO - PAFilter: Removed 118 hitters with < 37 PA (partial season)
15:00:59 - new_pipeline.common.transformers.age_enricher - INFO - AgeEnricher: Loaded Age for 673 hitters
15:00:59 - new_pipeline.common.transformers.age_enricher - INFO - AgeEnricher: Added Age column (range: 21-41)
15:00:59 - new_pipeline.common.transformers.hitter_features - INFO - Loading hitter features...


Loading 2025 current season data...
Loading partial season data: fangraphs_hitters_2025_firsthalf.csv
Loaded 606 hitters (raw)

Processing through pipeline...


15:00:59 - new_pipeline.common.transformers.hitter_features - INFO - Loaded 11 hitter feature sets (33 total columns)
15:00:59 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Learned replacement values for 28 features
15:00:59 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Imputed 99 missing values
15:00:59 - new_pipeline.common.transformers.validators - WARNING - FeatureValidator found issues:
  - Feature 'AVG' range [0.07, 0.36] outside expected [0.1, 0.4]
  - Feature 'OBP' range [0.12, 0.47] outside expected [0.2, 0.5]
  - Feature 'SLG' range [0.07, 0.74] outside expected [0.2, 0.8]
15:00:59 - new_pipeline.common.transformers.feature_selector - INFO - FeatureSelector: Selected 9 features + 10 metadata columns
15:00:59 - new_pipeline.common.transformers.normalizers - INFO - WARNormalizer: Added 'WAR_per_600' column


Processed 488 qualified hitters


In [27]:
# Cell 9: Generate 2025 Predictions and Display

print("Generating 2025 predictions...")

hitter_predictions = generate_predictions(
    hitter_2025_processed,
    hitter_model,
    player_type='hitter'
)

print(f"\nGenerated predictions for {len(hitter_predictions)} hitters")

# Calculate prediction error
hitter_predictions['Error'] = hitter_predictions['WAR'] - hitter_predictions['Predicted_Current_WAR']

# Display top 10 hitters by actual current WAR
print("\n" + "="*90)
print("TOP 10 CURRENT SEASON PREDICTIONS")
print("="*90)
print()

top_10 = hitter_predictions.nlargest(10, 'WAR')

# Select columns for display
display_cols = [
    'Name', 'Team', 'PA',
    'WAR', 'Predicted_Current_WAR', 'Error'
]

top_10_display = top_10[display_cols].copy()

# Round for display
for col in ['WAR', 'Predicted_Current_WAR', 'Error']:
    top_10_display[col] = top_10_display[col].round(1)

# Rename columns for cleaner display
top_10_display.columns = ['Name', 'Team', 'PA', 'Actual', 'Predicted', 'Error']

print(top_10_display.to_string(index=False))
print()
print("="*90)
print()
print("Column Guide:")
print("  Actual = Actual Current_WAR from data")
print("  Predicted = Predicted Current_WAR from model (tier-based blending)")
print("  Error = Actual - Predicted (positive = model underestimated)")
print()
print("Note: Full season projections (Current + ROS) available in oWAR_overview.ipynb")
print("="*90)

c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning:

X has feature names, but StandardScaler was fitted without feature names



Generating 2025 predictions...

Generated predictions for 488 hitters

TOP 10 CURRENT SEASON PREDICTIONS

               Name Team  PA  Actual  Predicted  Error
        Aaron Judge  NYY 429     7.0        6.3    0.6
        Cal Raleigh  SEA 417     6.2        6.7   -0.5
     Bobby Witt Jr.  KCR 420     4.8        4.4    0.3
Pete Crow-Armstrong  CHC 401     4.6        4.4    0.2
      Shohei Ohtani  LAD 438     4.1        5.0   -0.9
        Jeremy Peña  HOU 350     4.0        3.5    0.5
        Trea Turner  PHI 423     3.8        3.2    0.6
        Kyle Tucker  CHC 423     3.8        4.5   -0.7
       José Ramírez  CLE 391     3.8        4.5   -0.8
       Byron Buxton  MIN 333     3.7        4.4   -0.6


Column Guide:
  Actual = Actual Current_WAR from data
  Predicted = Predicted Current_WAR from model (tier-based blending)
  Error = Actual - Predicted (positive = model underestimated)

Note: Full season projections (Current + ROS) available in oWAR_overview.ipynb


In [28]:
# Cell 9.5: Diagnostic Analysis - Elite Hitter Tier Classification

print("="*90)
print("DIAGNOSTIC: Tier Classification and Quantile Analysis")
print("="*90)
print()

# Get top 20 hitters by actual WAR to analyze
top_20 = hitter_predictions.nlargest(20, 'WAR')

# Extract features for top 20
X_top20 = top_20[HITTER_MODEL_FEATURES].values

# Detect season progress from data
avg_pa = top_20['PA'].mean()
season_pct = min(avg_pa / 600, 1.0)
print(f"Season Progress: {season_pct:.1%} (avg PA: {avg_pa:.1f})")
print()

# Scale features
X_scaled = hitter_model.scaler.transform(X_top20)

# Get all predictions (these are WAR_per_600 rates)
et_pred = hitter_model.extratrees_model.predict(X_scaled)
keras_quantiles = hitter_model.keras_model.predict(X_scaled, verbose=0)
histgb_quantiles = hitter_model.histgb_model.get_quantile_predictions(X_scaled)

# Calculate initial estimate (used for tier classification)
keras_q50 = keras_quantiles[:, 0]
initial = 0.4 * et_pred + 0.3 * keras_q50 + 0.3 * histgb_quantiles['q50']

# Get dynamic thresholds (now with role='hitter')
avg_threshold, elite_threshold = hitter_model._get_dynamic_thresholds(season_pct, 'hitter')
print(f"Thresholds: avg={avg_threshold:.2f}, elite={elite_threshold:.2f} (WAR_per_600 rates)")

# Calculate cumulative threshold ranges for displayed group
player_pa = top_20['PA'].values
min_pa = player_pa.min()
max_pa = player_pa.max()
avg_min_cumul = avg_threshold * (min_pa / 600)
avg_max_cumul = avg_threshold * (max_pa / 600)
elite_min_cumul = elite_threshold * (min_pa / 600)
elite_max_cumul = elite_threshold * (max_pa / 600)

print(f"Cumulative ranges (for PA {min_pa:.0f}-{max_pa:.0f}): avg={avg_min_cumul:.2f}-{avg_max_cumul:.2f}, elite={elite_min_cumul:.2f}-{elite_max_cumul:.2f}")
print()

# Classify tiers
tier_labels = np.array([
    'average' if w < avg_threshold 
    else 'good' if w < elite_threshold 
    else 'elite' 
    for w in initial
])

# Get final blended predictions (still rates)
final = hitter_model._blend_predictions(et_pred, keras_quantiles, histgb_quantiles)

# Convert all rate predictions to cumulative WAR (use PA/600)
initial_cumul = initial * (player_pa / 600)
et_cumul = et_pred * (player_pa / 600)
k_q50_cumul = keras_quantiles[:, 0] * (player_pa / 600)
k_q75_cumul = keras_quantiles[:, 1] * (player_pa / 600)
k_q90_cumul = keras_quantiles[:, 2] * (player_pa / 600)
h_q90_cumul = histgb_quantiles['q90'] * (player_pa / 600)
final_cumul = final * (player_pa / 600)

# Build diagnostic dataframe (all cumulative WAR)
player_names = top_20['Name'].values
player_actual = top_20['WAR'].values

diag_df = pd.DataFrame({
    'Name': player_names,
    'PA': np.round(player_pa, 0),
    'Actual': np.round(player_actual, 1),
    'Tier': tier_labels,
    'Initial': np.round(initial_cumul, 1),
    'ET': np.round(et_cumul, 1),
    'K_q50': np.round(k_q50_cumul, 1),
    'K_q75': np.round(k_q75_cumul, 1),
    'K_q90': np.round(k_q90_cumul, 1),
    'H_q90': np.round(h_q90_cumul, 1),
    'Final': np.round(final_cumul, 1),
    'Error': np.round(player_actual - final_cumul, 1)
})

print("HITTER DIAGNOSTICS (All values are cumulative WAR):")
print(diag_df.to_string(index=False))
print()

# Summary statistics
elite_count = (tier_labels == 'elite').sum()
good_count = (tier_labels == 'good').sum()
avg_count = (tier_labels == 'average').sum()

print(f"Tier Distribution: Elite={elite_count}, Good={good_count}, Average={avg_count}")

if elite_count > 0:
    elite_mask = tier_labels == 'elite'
    print(f"Elite tier average q90 usage: 45% (30% keras + 15% histgb)")
    print(f"Elite tier avg error: {(player_actual[elite_mask] - final_cumul[elite_mask]).mean():.2f}")

print()
print("="*90)
print()
print("Analysis Questions:")
print("  1. Are elite hitters (Actual >= 3.5) being classified as 'elite' tier?")
print("  2. Are K_q90 values high enough to reach Actual values?")
print("  3. Is the error consistent (all underestimated) or random?")
print("  4. What's the gap between K_q90 and Actual for elite hitters?")
print()
print("Note: All prediction values now shown as cumulative WAR (converted from rates)")
print("="*90)

DIAGNOSTIC: Tier Classification and Quantile Analysis

Season Progress: 65.1% (avg PA: 390.8)

Thresholds: avg=1.30, elite=2.28 (WAR_per_600 rates)
Cumulative ranges (for PA 297-438): avg=0.64-0.95, elite=1.13-1.66

HITTER DIAGNOSTICS (All values are cumulative WAR):
               Name  PA  Actual  Tier  Initial  ET  K_q50  K_q75  K_q90  H_q90  Final  Error
        Aaron Judge 429     7.0 elite      5.1 4.7    6.1    6.9    7.9    4.9    6.3    0.6
        Cal Raleigh 417     6.2 elite      5.3 4.0    6.3    7.1    8.3    6.3    6.7   -0.5
     Bobby Witt Jr. 420     4.8 elite      3.8 3.3    4.1    4.4    5.1    4.7    4.4    0.3
Pete Crow-Armstrong 401     4.6 elite      3.3 2.8    4.0    4.5    5.3    4.6    4.4    0.2
      Shohei Ohtani 438     4.1 elite      4.3 3.9    4.5    5.1    5.8    4.8    5.0   -0.9
        Jeremy Peña 350     4.0 elite      3.1 2.8    3.1    3.4    4.0    3.7    3.5    0.5
        Trea Turner 423     3.8 elite      2.6 2.3    2.8    3.1    3.7    3.6   

In [29]:
# Cell 10: Actual vs Predicted Plot

# Add position for coloring if available
if 'Primary_Position' in hitter_processed.columns:
    color_by = hitter_processed['Primary_Position'].values
else:
    color_by = None

fig_scatter = create_actual_vs_predicted(
    y_true=y_train,
    y_pred=y_pred_train,
    color_by=color_by
)

fig_scatter.update_layout(title="Hitter WAR: Actual vs Predicted (Training Set)")
fig_scatter.show()

In [30]:
# DIAGNOSTIC: Check prediction scaling
print("DIAGNOSTIC: Checking prediction scaling")
print("="*90)
print()

sample = hitter_predictions.nlargest(10, 'Total_Projected_WAR')[[
    'Name', 'PA', 
    'Predicted_WAR_per_600', 'Predicted_Current_WAR', 
    'ROS_WAR', 'Total_Projected_WAR', 'WAR'
]].copy()

# Calculate what Predicted_Current_WAR SHOULD be if scaling correctly
sample['Expected_Current'] = sample['Predicted_WAR_per_600'] * (sample['PA'] / 600)
sample['Scaling_Ratio'] = sample['Predicted_Current_WAR'] / sample['Expected_Current']

# Round for readability
for col in ['Predicted_WAR_per_600', 'Predicted_Current_WAR', 'Expected_Current', 'WAR', 'ROS_WAR', 'Total_Projected_WAR']:
    sample[col] = sample[col].round(2)
sample['Scaling_Ratio'] = sample['Scaling_Ratio'].round(3)

print(sample.to_string(index=False))
print()
print("="*90)
print("Analysis:")
print("  - Predicted_WAR_per_600 = Model's rate prediction (per 600 PA)")
print("  - Expected_Current = What Predicted_Current_WAR should be: Predicted_WAR_per_600 × (PA / 600)")
print("  - Scaling_Ratio = Predicted_Current_WAR / Expected_Current")
print()
print("If Scaling_Ratio = 1.000, scaling is correct")
print("If Scaling_Ratio != 1.000, then full_season_usage is NOT 600 in generate_predictions()")
print("="*90)

DIAGNOSTIC: Checking prediction scaling

               Name  PA  Predicted_WAR_per_600  Predicted_Current_WAR  ROS_WAR  Total_Projected_WAR  WAR  Expected_Current  Scaling_Ratio
        Cal Raleigh 417                   9.65                   6.70     1.06                 7.76 6.17              6.70            1.0
        Aaron Judge 429                   8.88                   6.35     0.98                 7.32 6.99              6.35            1.0
      Shohei Ohtani 438                   6.88                   5.02     0.77                 5.79 4.12              5.02            1.0
       José Ramírez 391                   6.94                   4.52     0.81                 5.33 3.77              4.52            1.0
        Kyle Tucker 423                   6.36                   4.49     0.71                 5.20 3.78              4.49            1.0
Pete Crow-Armstrong 401                   6.58                   4.40     0.73                 5.13 4.57              4.40         

In [31]:
# Cell 11: Residual Analysis

residuals = y_train - y_pred_train

if 'Primary_Position' in hitter_processed.columns:
    color_by = hitter_processed['Primary_Position'].values
else:
    color_by = None

fig_residuals = create_residual_plot(
    residuals=residuals,
    color_by=color_by
)

fig_residuals.update_layout(title="Hitter Residual Distribution")
fig_residuals.show()

print(f"\nResidual statistics:")
print(f"  Mean: {residuals.mean():.3f}")
print(f"  Std: {residuals.std():.3f}")
print(f"  Min: {residuals.min():.3f}")
print(f"  Max: {residuals.max():.3f}")


Residual statistics:
  Mean: -0.226
  Std: 1.038
  Min: -5.058
  Max: 5.410


In [32]:
# Cell 13: Error Analysis by Position

if 'Primary_Position' in hitter_processed.columns:
    print("Analyzing errors by position...")
    
    # Filter to only players with valid positions (exclude NaN)
    has_position = hitter_processed['Primary_Position'].notna()
    n_with_position = has_position.sum()
    n_without = (~has_position).sum()
    
    if n_without > 0:
        print(f"Note: Excluding {n_without} hitters with missing position data")
    
    position_errors = analyze_errors_by_group(
        residuals=residuals[has_position],
        groups=hitter_processed.loc[has_position, 'Primary_Position'].values
    )
    
    print("\n" + "="*50)
    print("ERROR ANALYSIS BY POSITION")
    print("="*50)
    for position, metrics in position_errors.items():
        print(f"\n{position}:")
        print(f"  Count: {metrics['count']}")
        print(f"  MAE: {metrics['MAE']:.3f}")
        print(f"  RMSE: {metrics['RMSE']:.3f}")
        print(f"  Mean Error: {metrics['mean_error']:.3f}")
        print(f"  Std Error: {metrics['std_error']:.3f}")
    print("="*50)
else:
    print("\nPosition information not available for error analysis")
    print("Note: All errors analyzed together since unified model used")

Analyzing errors by position...
Note: Excluding 2 hitters with missing position data

ERROR ANALYSIS BY POSITION

1B:
  Count: 361
  MAE: 0.653
  RMSE: 0.834
  Mean Error: -0.349
  Std Error: 0.757

2B:
  Count: 173
  MAE: 0.861
  RMSE: 1.107
  Mean Error: -0.494
  Std Error: 0.991

3B:
  Count: 408
  MAE: 0.798
  RMSE: 1.010
  Mean Error: -0.393
  Std Error: 0.931

C:
  Count: 533
  MAE: 1.076
  RMSE: 1.366
  Mean Error: -0.294
  Std Error: 1.334

CF:
  Count: 112
  MAE: 0.744
  RMSE: 0.995
  Mean Error: 0.295
  Std Error: 0.950

LF:
  Count: 370
  MAE: 0.806
  RMSE: 1.006
  Mean Error: -0.412
  Std Error: 0.917

P:
  Count: 243
  MAE: 0.877
  RMSE: 1.165
  Mean Error: -0.181
  Std Error: 1.151

RF:
  Count: 1292
  MAE: 0.756
  RMSE: 0.981
  Mean Error: -0.100
  Std Error: 0.975

SS:
  Count: 742
  MAE: 0.840
  RMSE: 1.069
  Mean Error: -0.182
  Std Error: 1.053


In [33]:
# Cell 14: Enhanced Feature Analysis

print("Analyzing enhanced features...")

# Check if enhanced features are in the data
enhanced_features = ['Enhanced_Baserunning', 'Enhanced_Defense', 'Positional_WAR']
available_enhanced = [f for f in enhanced_features if f in HITTER_MODEL_FEATURES]

if available_enhanced:
    print(f"\nEnhanced features in model: {available_enhanced}")
    
    # Show distribution of enhanced features
    for feat in available_enhanced:
        feat_idx = HITTER_MODEL_FEATURES.index(feat)
        feat_values = X_train[:, feat_idx]
        print(f"\n{feat}:")
        print(f"  Range: {feat_values.min():.3f} to {feat_values.max():.3f}")
        print(f"  Mean: {feat_values.mean():.3f}")
        print(f"  Std: {feat_values.std():.3f}")
else:
    print("\nNo enhanced features found in model")

Analyzing enhanced features...

Enhanced features in model: ['Enhanced_Baserunning', 'Enhanced_Defense', 'Positional_WAR']

Enhanced_Baserunning:
  Range: -4.632 to 9.038
  Mean: 0.350
  Std: 1.394

Enhanced_Defense:
  Range: -8.553 to 30.000
  Mean: 6.204
  Std: 8.223

Positional_WAR:
  Range: -1.250 to 1.250
  Mean: -0.003
  Std: 0.723


In [34]:
# Cell 15: Save Model and Predictions

# Save model using proper method (handles Keras + sklearn correctly)
# Use absolute path with project_root (defined in Cell 1)
model_base_path = str(project_root / 'models' / 'hitter_ensemble_2025')
hitter_model.save(model_base_path)
print(f"Model saved to: {model_base_path}")
print("  Created files:")
print("    - hitter_ensemble_2025.pkl")
print("    - hitter_ensemble_2025_keras.keras")

# Save predictions
predictions_path = project_root / 'predictions' / 'hitter_predictions_2025.csv'
predictions_path.parent.mkdir(exist_ok=True)
hitter_predictions.to_csv(predictions_path, index=False)
print(f"\nPredictions saved to: {predictions_path}")

print("\n" + "="*50)
print("HITTER PIPELINE COMPLETE!")
print("="*50)
print(f"Trained on {len(hitter_processed)} historical hitter-seasons")
print(f"Generated predictions for {len(hitter_predictions)} 2025 hitters")
print(f"Overall MAE: {metrics['MAE']:.3f}")
# print(f"Overall R²: {metrics['R²']:.3f}")
print(f"\nUnified Model: Single ensemble for all positions")
print("Position differences handled by Positional_WAR feature")
print("="*50)

Model saved to: c:\Users\nairs\Documents\GithubProjects\oWAR\models\hitter_ensemble_2025
  Created files:
    - hitter_ensemble_2025.pkl
    - hitter_ensemble_2025_keras.keras

Predictions saved to: c:\Users\nairs\Documents\GithubProjects\oWAR\predictions\hitter_predictions_2025.csv

HITTER PIPELINE COMPLETE!
Trained on 4236 historical hitter-seasons
Generated predictions for 488 2025 hitters
Overall MAE: 0.840

Unified Model: Single ensemble for all positions
Position differences handled by Positional_WAR feature
